In [ ]:
import pandas as pd
from pandas.errors import PerformanceWarning
import warnings
import sys
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

warnings.simplefilter('ignore', category=PerformanceWarning)
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
print(sys.path)

In [ ]:
from src.features.load_day_ahead_dataset import load_day_ahead_dataset

start = pd.Timestamp("2024-07-01")
end = pd.Timestamp("2025-04-01")
val_cutoff = pd.Timestamp("2025-01-01")
select_cutoff = pd.Timestamp("2025-02-01")
test_cutoff = pd.Timestamp("2025-03-01")

df = load_day_ahead_dataset(start, end)

In [ ]:
from src.models.metrics import get_tabular_metrics, get_timeseries_metrics

ts_metrics = get_timeseries_metrics()
tab_metrics = get_tabular_metrics()

In [ ]:
from src.models.data_split import tabular_split, time_series_split, DA_KNOWN, DA_TARGET
from src.models.train_day_ahead_time_series import train_day_ahead_time_series
from src.models.train_day_ahead_tabular import train_day_ahead_tabular
from src.models.evaluation import evaluate_timeseries_model, evaluate_tabular_model

tab_df = df[[col for col in df.columns if any(col.startswith(prefix) for prefix in DA_KNOWN)] + [DA_TARGET]]
tab_train, tab_val, tab_select, tab_test = tabular_split(tab_df, val_cutoff, select_cutoff, test_cutoff)
ts_train, ts_val, ts_select, ts_test = time_series_split(df, val_cutoff, select_cutoff, test_cutoff)

In [ ]:
ts_mae_model = train_day_ahead_time_series(ts_train, ts_val, "MAE", cache="MAE")
display(evaluate_timeseries_model(ts_mae_model, ts_train, ts_val, ts_select, ts_metrics))

In [ ]:
tab_mae_model = train_day_ahead_tabular(tab_train, tab_val, "mae", cache="MAE")
display(evaluate_tabular_model(tab_mae_model, tab_train, tab_val, tab_select, tab_metrics))

In [ ]:
# from autogluon.timeseries import TimeSeriesDataFrame
#
# ts_retrain = TimeSeriesDataFrame.from_data_frame(pd.concat([ts_train, ts_val, ts_select]))
# ts_model = train_day_ahead_time_series(ts_retrain, None, "MAE", cache="retrain", models={"Chronos": {}})
# display(ts_model.leaderboard(ts_test, extra_metrics=ts_metrics))